In [7]:
import numpy as np
from sklearn.datasets import load_files
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score

In [8]:
path_train = '/Users/mikolajkrupinski/scikit_learn_data/20news_home/20news-bydate-train'
path_test =  '/Users/mikolajkrupinski/scikit_learn_data/20news_home/20news-bydate-test'

try:
    data_train = load_files(path_train, encoding='latin1')
    data_test = load_files(path_test, encoding='latin1')
except FileNotFoundError:
    print("[ERROR] Dataset not found. Please download the 20 Newsgroups dataset from scikit-learn's website and place it in the specified directory.")
    raise

X_train, y_train = data_train.data, data_train.target
X_test, y_test = data_test.data, data_test.target

In [9]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')), 
    ('clf', LinearSVC(dual='auto', random_state=42))
])

param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__sublinear_tf': [True, False],
    'clf__C': [0.1, 1, 10],
}

grid_search = GridSearchCV(
    pipeline, 
    param_grid, 
    cv=3, 
    n_jobs=-1, 
    verbose=1,
    scoring='accuracy'
)

In [10]:
grid_search.fit(X_train, y_train)
print(f"Best parameters found on train data set: {grid_search.best_params_}")

Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best parameters found on train data set: {'clf__C': 10, 'tfidf__ngram_range': (1, 2), 'tfidf__sublinear_tf': True}


In [12]:
best_model = grid_search.best_estimator_

kfold_test = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
test_scores = cross_val_score(best_model, X_test, y_test, cv=kfold_test, n_jobs=-1)

final_accuracy = np.mean(test_scores) * 100
print(f"Results for individual folds on test data set: {test_scores}")
print(f"Final average accuracy on test data set: {final_accuracy:.2f}%")

Results for individual folds on test data set: [0.93231586 0.91838089 0.9309429  0.93359894 0.92231076]
Final average accuracy on test data set: 92.75%
